In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
FEATURE_DIR = PROJECT_ROOT / "data" / "features"

full_features = pd.read_parquet(FEATURE_DIR / "full_features_mirrored.parquet")
print(full_features.shape)

match_ids = full_features['match_id'].unique()
print(f"Total unique matches: {len(match_ids)}")

rng = np.random.default_rng(seed=42)
rng.shuffle(match_ids)

n = len(match_ids)
train_ids = match_ids[:int(n * 0.7)]
val_ids = match_ids[int(n * 0.7):int(n * 0.85)] #validation
test_ids = match_ids[int(n * 0.85):]

print(f"Train matches: {len(train_ids)}")
print(f"Val matches: {len(val_ids)}")
print(f"Test matches: {len(test_ids)}")

train_df = full_features[full_features['match_id'].isin(train_ids)].copy()
val_df = full_features[full_features['match_id'].isin(val_ids)].copy()
test_df = full_features[full_features['match_id'].isin(test_ids)].copy()

print(f"\nTrain rows: {train_df.shape[0]}")
print(f"Val rows: {val_df.shape[0]}")
print(f"Test rows: {test_df.shape[0]}")

(178662, 27)
Total unique matches: 10000
Train matches: 7000
Val matches: 1500
Test matches: 1500

Train rows: 124866
Val rows: 26896
Test rows: 26900


In [4]:
exclude_cols = ['match_id', 'game_time_s', 'perspective', 'won', 'duration_s']
feature_cols = [c for c in full_features.columns if c not in exclude_cols]
print("Feature columns:", feature_cols)

Feature columns: ['team', 'team_net_worth_team0', 'team_kills_team0', 'team_deaths_team0', 'team_damage_team0', 'own_deaths_cumulative', 'enemy_deaths_cumulative', 'own_objectives_cumulative', 'enemy_objectives_cumulative', 'own_mid_boss_claims', 'enemy_mid_boss_claims', 'team_net_worth_team1', 'team_kills_team1', 'team_deaths_team1', 'team_damage_team1', 'net_worth_diff', 'kills_diff', 'deaths_diff', 'damage_diff', 'objectives_diff', 'deaths_cumulative_diff', 'mid_boss_diff']


In [6]:
exclude_cols = [
    'match_id', 'game_time_s', 'perspective', 'won', 'duration_s', 'team',
    'team_net_worth_team0', 'team_net_worth_team1',
    'team_kills_team0', 'team_kills_team1',
    'team_deaths_team0', 'team_deaths_team1',
    'team_damage_team0', 'team_damage_team1',
]

feature_cols = [c for c in full_features.columns if c not in exclude_cols]
print("Feature columns:", feature_cols)
print(f"\nTotal features: {len(feature_cols)}")

Feature columns: ['own_deaths_cumulative', 'enemy_deaths_cumulative', 'own_objectives_cumulative', 'enemy_objectives_cumulative', 'own_mid_boss_claims', 'enemy_mid_boss_claims', 'net_worth_diff', 'kills_diff', 'deaths_diff', 'damage_diff', 'objectives_diff', 'deaths_cumulative_diff', 'mid_boss_diff']

Total features: 13


In [11]:
import torch
from torch.nn.utils.rnn import pad_sequence

def build_sequences(df, feature_cols):
    sequences = []
    lengths = []
    labels = []
    group_keys = []

    grouped = df.sort_values('game_time_s').groupby(['match_id', 'perspective'])

    for (match_id, perspective), group in grouped:
        seq = group[feature_cols].values.astype(np.float32)
        sequences.append(torch.tensor(seq))
        lengths.append(len(seq))
        labels.append(bool(group['won'].iloc[0]))
        group_keys.append((match_id, perspective))

    padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)
    lengths = torch.tensor(lengths)
    labels = torch.tensor(labels, dtype=torch.float32)

    return padded, lengths, labels, group_keys

print("Building train sequences")
X_train, len_train, y_train, keys_train = build_sequences(train_df, feature_cols)
print(f"X_train shape: {X_train.shape}")  
print(f"len_train shape: {len_train.shape}")
print(f"y_train shape: {y_train.shape}")

print("\nBuilding val sequences")
X_val, len_val, y_val, keys_val = build_sequences(val_df, feature_cols)
print(f"X_val shape: {X_val.shape}")

print("\nBuilding test sequences")
X_test, len_test, y_test, keys_test = build_sequences(test_df, feature_cols)
print(f"X_test shape: {X_test.shape}")

Building train sequences
X_train shape: torch.Size([14000, 14, 13])
len_train shape: torch.Size([14000])
y_train shape: torch.Size([14000])

Building val sequences
X_val shape: torch.Size([3000, 14, 13])

Building test sequences
X_test shape: torch.Size([3000, 14, 13])


In [14]:
MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_ready"
MODEL_DATA_DIR.mkdir(parents=True, exist_ok=True)

torch.save({
    'X_train': X_train, 'len_train': len_train, 'y_train': y_train,
    'X_val': X_val, 'len_val': len_val, 'y_val': y_val,
    'X_test': X_test, 'len_test': len_test, 'y_test': y_test,
    'feature_cols': feature_cols,
}, MODEL_DATA_DIR / "sequences.pt")

print(f"Saved to {MODEL_DATA_DIR / 'sequences.pt'}")

Saved to C:\.STUFF\coding\dl-match-analyser\data\model_ready\sequences.pt


In [16]:
import torch.nn as nn

class WinProbabilityLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: [batch, seq_len, input_size]
        lstm_out, _ = self.lstm(x)  # lstm_out: [batch, seq_len, hidden_size]
        logits = self.fc(lstm_out)  # [batch, seq_len, 1]
        return logits.squeeze(-1)   # [batch, seq_len]

model = WinProbabilityLSTM(input_size=len(feature_cols))
print(model)

# quick sanity check: run one batch through untrained model
sample_out = model(X_train[:4])
print("\nSample output shape:", sample_out.shape)  # should be [4, 14]
print("Sample output (untrained, meaningless yet):")
print(sample_out)


WinProbabilityLSTM(
  (lstm): LSTM(13, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

Sample output shape: torch.Size([4, 14])
Sample output (untrained, meaningless yet):
tensor([[-9.1152e-02,  2.4876e-01,  3.3841e-01,  1.2020e-01,  1.4039e-01,
         -1.7977e-01,  2.4836e-01,  2.5223e-01, -2.1147e-01, -1.8446e-01,
         -1.2079e-01, -5.6211e-02,  5.7378e-03,  5.5159e-02],
        [ 1.1866e-01,  2.9315e-01,  1.2642e-01,  2.2959e-01,  1.6657e-01,
          1.6995e-01,  1.6201e-01,  4.1410e-01,  1.2049e-01,  1.1417e-01,
          1.3844e-01,  1.4760e-01,  1.4327e-01,  1.3972e-01],
        [ 2.4339e-01,  2.5831e-01,  2.7227e-01,  2.7428e-01,  2.8122e-01,
          1.8043e-01,  1.9708e-01, -5.1081e-02, -5.8791e-03,  2.9391e-02,
          6.7497e-02,  9.7396e-02,  1.1622e-01,  1.2680e-01],
        [ 2.8225e-01,  2.1482e-01,  2.1661e-01,  2.1493e-01,  6.0884e-02,
          1.8082e-01,  2.1196e-01,  7.4711e-02, -2.7783e-04, -1.9955e-02,
         -2.483

In [18]:
import torch.optim as optim

def make_mask(lengths, max_len):
    # mask[i, t] = 1 if timestep t is real data for sequence i, 0 if padding
    return (torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)).float()

max_len = X_train.shape[1]
mask_train = make_mask(len_train, max_len)
mask_val = make_mask(len_val, max_len)

# expand y (one label per sequence) to one label per timestep
y_train_expanded = y_train.unsqueeze(1).expand(-1, max_len)
y_val_expanded = y_val.unsqueeze(1).expand(-1, max_len)

model = WinProbabilityLSTM(input_size=len(feature_cols))
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss(reduction='none')  # 'none' so we can apply the mask manually

def masked_loss(logits, targets, mask):
    losses = criterion(logits, targets)         # [batch, seq_len]
    losses = losses * mask                       # zero out padded positions
    return losses.sum() / mask.sum()             # average over real timesteps only

n_epochs = 20
batch_size = 128
n_train = X_train.shape[0]

for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(n_train)
    total_loss = 0.0

    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        xb = X_train[idx]
        yb = y_train_expanded[idx]
        mb = mask_train[idx]

        optimizer.zero_grad()
        logits = model(xb)
        loss = masked_loss(logits, yb, mb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(idx)

    train_loss = total_loss / n_train

    # validation
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = masked_loss(val_logits, y_val_expanded, mask_val).item()

    print(f"Epoch {epoch+1}/{n_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

Epoch 1/20 - train_loss: 0.6556 - val_loss: 0.6271
Epoch 2/20 - train_loss: 0.6049 - val_loss: 0.6188
Epoch 3/20 - train_loss: 0.5985 - val_loss: 0.6172
Epoch 4/20 - train_loss: 0.5971 - val_loss: 0.6167
Epoch 5/20 - train_loss: 0.5956 - val_loss: 0.6157
Epoch 6/20 - train_loss: 0.5944 - val_loss: 0.6146
Epoch 7/20 - train_loss: 0.5939 - val_loss: 0.6139
Epoch 8/20 - train_loss: 0.5921 - val_loss: 0.6121
Epoch 9/20 - train_loss: 0.5920 - val_loss: 0.6106
Epoch 10/20 - train_loss: 0.5910 - val_loss: 0.6100
Epoch 11/20 - train_loss: 0.5898 - val_loss: 0.6105
Epoch 12/20 - train_loss: 0.5898 - val_loss: 0.6082
Epoch 13/20 - train_loss: 0.5877 - val_loss: 0.6072
Epoch 14/20 - train_loss: 0.5859 - val_loss: 0.6091
Epoch 15/20 - train_loss: 0.5859 - val_loss: 0.6033
Epoch 16/20 - train_loss: 0.5860 - val_loss: 0.6054
Epoch 17/20 - train_loss: 0.5840 - val_loss: 0.6024
Epoch 18/20 - train_loss: 0.5843 - val_loss: 0.5997
Epoch 19/20 - train_loss: 0.5822 - val_loss: 0.6039
Epoch 20/20 - train_l

In [20]:
def compute_final_timestep_accuracy(model, X, lengths, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)  
        probs = torch.sigmoid(logits)

        batch_size = X.shape[0]
        final_idx = lengths - 1  
        final_probs = probs[torch.arange(batch_size), final_idx]

        preds = (final_probs > 0.5).float()
        acc = (preds == y).float().mean().item()
    return acc, final_probs

train_acc, _ = compute_final_timestep_accuracy(model, X_train, len_train, y_train)
val_acc, _ = compute_final_timestep_accuracy(model, X_val, len_val, y_val)

print(f"Train accuracy (final timestep): {train_acc:.4f}")
print(f"Val accuracy (final timestep): {val_acc:.4f}")

Train accuracy (final timestep): 0.9514
Val accuracy (final timestep): 0.9513


In [22]:
# check: does a trivial rule (net_worth_diff > 0 => win) already get ~95%?
final_idx_train = len_train - 1
final_net_worth_diff = X_train[torch.arange(X_train.shape[0]), final_idx_train, feature_cols.index('net_worth_diff')]

trivial_preds = (final_net_worth_diff > 0).float()
trivial_acc = (trivial_preds == y_train).float().mean().item()
print(f"Trivial rule (net_worth_diff > 0) train accuracy: {trivial_acc:.4f}")

Trivial rule (net_worth_diff > 0) train accuracy: 0.9591


In [23]:
def accuracy_by_timestep(model, X, lengths, y, feature_cols, max_len):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)

    results = []
    for t in range(max_len):
        valid = lengths > t  
        if valid.sum() == 0:
            continue
        preds_t = (probs[valid, t] > 0.5).float()
        y_t = y[valid]
        acc_t = (preds_t == y_t).float().mean().item()
        results.append({'timestep': t, 'n_matches': valid.sum().item(), 'accuracy': acc_t})

    return pd.DataFrame(results)

acc_by_t = accuracy_by_timestep(model, X_val, len_val, y_val, feature_cols, max_len)
print(acc_by_t)

    timestep  n_matches  accuracy
0          0       3000  0.594333
1          1       3000  0.614667
2          2       3000  0.626333
3          3       3000  0.642000
4          4       3000  0.653000
5          5       2970  0.718182
6          6       2792  0.738539
7          7       2392  0.768395
8          8       1788  0.766219
9          9       1148  0.781359
10        10        550  0.818182
11        11        190  0.831579
12        12         52  0.865385
13        13         14  0.857143
